In [ ]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
detect = YOLO('yolo.pt')

In [ ]:
results = detect.predict(r"F:\Datasets\Reddit-fishes\aquarium_top_1\gallery-dl\reddit\Aquariums\1ay10y3 This is Ziggy, my ancient zebra danio, and tonight is his last night with me.png", conf=0.5, iou=0.1)

for result in results:
    im_array = result.plot()  # Это numpy array с нарисованными bbox'ами
    im = Image.fromarray(im_array[..., ::-1])  # Конвертируем BGR -> RGB
    plt.figure(figsize=(12, 8))
    plt.imshow(im)
    plt.axis('off')
    plt.show()

In [ ]:
for result in results:
    print("Обнаружены объекты:")
    for box in result.boxes:
        class_id = int(box.cls)  # ID класса
        confidence = float(box.conf)  # Уверенность
        class_name = result.names[class_id]  # Название класса
        print(f"- {class_name}: {confidence:.2f}")

In [ ]:
from mytemplates.ML.predict import yolo_predict

yolo_predict(
    model_path = 'runs/detect/fish_tiny_aug/weights/best.pt',
    source='Dataset_hand_label/valid/images',
    device=0,
    save_txt=True,
    save_conf=True,
    conf=0.45,
    project="runs/autolabel",
    name='fullset_conf045_val',
    exist_ok=True  # не создавать новую папку, если уже есть
)

---

Filter images with my YOLO model

In [ ]:
import os
import shutil
from ultralytics import YOLO
import cv2

# Пути
source_dir = r"F:\Datasets\Reddit-fishes\aquarium_new\gallery-dl\reddit\Aquariums"
good_dir = r"F:\Datasets\Reddit-fishes\aquarium_new\good_fish"
bad_dir = r"F:\Datasets\Reddit-fishes\aquarium_new\bad_no_fish"
labels_dir = os.path.join(good_dir, "labels")  # Папка для разметки
processed_log = r"F:\Datasets\Reddit-fishes\aquarium_new\processed_files.txt"

# Создаем папки
os.makedirs(good_dir, exist_ok=True)
os.makedirs(bad_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)  # Создаем папку для labels

# Загружаем лог
processed_files = set()
if os.path.exists(processed_log):
    with open(processed_log, 'r') as f:
        processed_files = set(f.read().splitlines())

model = YOLO('./nout.pt')

with open(processed_log, 'a', encoding='utf-8') as log_file:
    for filename in os.listdir(source_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            if filename in processed_files:
                print(f"⏩ Already processed: {filename}")
                continue
            
            img_path = os.path.join(source_dir, filename)
            results = model(img_path, conf=0.5, iou=0.1)
            
            if len(results[0].boxes) > 0:
                # Копируем изображение в good_dir
                shutil.copy2(img_path, good_dir)
                
                # Сохраняем разметку в формате YOLO
                img = cv2.imread(img_path)
                h, w, _ = img.shape
                
                label_filename = os.path.splitext(filename)[0] + '.txt'
                label_path = os.path.join(labels_dir, label_filename)
                
                with open(label_path, 'w') as label_file:
                    for box in results[0].boxes:
                        class_id = int(box.cls.item())
                        x_center, y_center, width, height = box.xywhn[0].tolist()
                        # Записываем в формат: class_id x_center y_center width height
                        label_file.write(f"{class_id} {x_center} {y_center} {width} {height}\n")
                
                message = f"✅ Fish found: {filename} (bboxes saved)"
            else:
                shutil.copy2(img_path, bad_dir)
                message = f"❌ No fish: {filename}"
            
            log_file.write(filename + '\n')
            print(message)

print("Фильтрация и разметка завершены!")